# 3. Computing labels and features, and cutting off the holdout

| | |
| --- | --- |
| **Purpose** | Add the candidate labels (what we want to predict) and features (what we predict with) to the universe, then cut the data into **pre-holdout** (all research) and **holdout** (final exam) |
| **Input** | The frozen universe copy from notebook 1 |
| **Outputs** | `data/hyperliquid/case_study_data/pre_holdout.parquet` and `holdout.parquet`: universe rows only, with the `fwd_*` labels and the `past_log_return_*` features |
| **Next notebook** | `4_feature_evaluation` |
| **Workflow** | Steps 5 and 6 of `docs/9-research_workflow.md` |

**The order matters, and it is chosen to avoid look-ahead:**

1. Compute labels and features on the **complete** series of every symbol (all rows, also the rows outside the universe), because a 7-day label needs the price 7 days later and a 30-day feature needs the price 30 days earlier, whether or not the coin was in the universe on those days.
2. Only then cut off the holdout, and filter to the universe rows.
3. Run every descriptive check **after** the cut and on the **pre-holdout rows only**, so that nothing about the holdout is looked at while we develop.

In [10]:
from pathlib import Path
import pandas as pd

from wh_research.data.hyperliquid.features import add_past_returns
from wh_research.data.hyperliquid.labels import add_forward_returns
from wh_research.evaluation.checks import check_saved_columns, check_split
from wh_research.evaluation.splits import find_holdout_start, split_holdout

## Load the universe

The frozen copy keeps **every** row with its `in_universe` flag.

In [11]:
PROJECT_ROOT = Path.cwd().parent.parent
data_path = PROJECT_ROOT/"data/hyperliquid/case_study_data/universe_1d_min_mdv_2m_exit_1m_280323_171026.parquet"
df = pd.read_parquet(data_path)

# Computing labels

### I will choose the following labels as candidates

-  1-day forward log-return
-  3-day forward log-return
-  7-day forward log-return
-  1-day forward log-return direction
-  3-day forward log-return direction
-  7-day forward log-return direction

### Computing the labels

`add_forward_returns` (`src/wh_research/data/hyperliquid/labels.py`, documented in `docs/7-forward_returns.md`) adds, for each horizon n, three columns:
- `fwd_log_return_{n}`: log(close n days later / this close), from this bar's `close_time`
- `fwd_direction_{n}`: 1 if that return is above 0, 0 if not, and missing where the return is missing
- `fwd_end_time_{n}`: when the label becomes known (used to purge the splits)

How it stays correct:
- The future close is looked up **by time** (`symbol`, `open_time + n days`), never by shifting rows, so a gap in a coin's series gives a missing label and never a wrong day. Coins never mix.
- Nothing is dropped or filled: a label is missing where the future bar does not exist (the last n days of every coin, and coins that stopped trading). The caller decides what to do with them.
- A missing direction stays missing and is not turned into "down".

We compute them on every row first and only then keep the rows where `in_universe` is True, so the labels never skip over days when a symbol was outside the universe. The `fwd_` columns look into the future by design: **they are never used as features.**

In [12]:
# Add the forward log-return, direction and end-time columns for 1, 3 and 7 days.
# Done on the complete series of every symbol (all rows), before filtering to the universe.
df = add_forward_returns(df, "1d", [1, 3, 7])

# Computing features

### I will choose the following features as candidates

-  7-day log-return
-  14-day log-return
-  30-day log-return

### Computing the features

`add_past_returns` (`src/wh_research/data/hyperliquid/features.py`, documented in `docs/8-past_returns.md`) adds, for each horizon n, `past_log_return_{n}`: log(close today / close n days ago).

It only uses the close at this bar's `close_time` and the close n days earlier, so it is known at the same moment as the `in_universe` flag and is used on the same row with no shift. Like the labels, it looks the earlier bar up by time, keeps all rows and leaves a missing value where the earlier bar does not exist. Like the labels, we compute them on every row first and only then keep the universe rows.

A 30-day feature needs 30 days of history. A coin only enters the universe on its 31st bar, so every universe row has the history it needs.

In [13]:
# Add the 7, 14 and 30-day past log-returns. Done on the complete series of every symbol (all rows), before filtering to the universe.
df = add_past_returns(df, "1d", [7, 14, 30])

# Cutting off the holdout

We use the same date boundary as in `2_determining_cv_folds` (cut by date: the last 30% of the days that have symbols in the universe are the holdout). `find_holdout_start` applies that rule, so the date is computed and not copied.

In [14]:
# every day with at least one symbol in the universe is counted, the last 30% of them are the holdout
holdout_start = find_holdout_start(df)

print(f"Holdout starts : {holdout_start}")

Holdout starts : 2025-09-22 00:00:00+00:00


### Purged split

The longest label is 7 days ahead, so `horizon = 7`. A pre-holdout row is kept only if its label is fully known (`fwd_end_time_7`) before the holdout starts, otherwise its label would look into the holdout. The 7 days just before the holdout are therefore in neither set.

`split_holdout` (`src/wh_research/evaluation/splits.py`) also keeps only the rows in the universe.

- `pre_holdout`: used for feature evaluation and all research. The CV folds are built from it later, after the features and the label are chosen
- `holdout`: everything from the holdout start. It is only for the final unbiased evaluation, so do not open it while developing models

Rows with a missing label are kept. The modelling step decides what to do with them.

In [15]:
pre_holdout, holdout = split_holdout(df, holdout_start, horizon=7)

sets = {"pre_holdout": pre_holdout, "holdout": holdout}

# a short summary of each set (only sizes and dates, nothing about prices or labels)
pd.DataFrame({
    name: {
        "rows": len(part),
        "symbols": part["symbol"].nunique(),
        "first open_time": part["open_time"].min(),
        "last open_time": part["open_time"].max(),
    }
    for name, part in sets.items()
}).T

,rows,symbols,first open_time,last open_time
pre_holdout,37725,143,2023-03-28 00:00:00+00:00,2025-09-14 00:00:00+00:00
holdout,21454,110,2025-09-22 00:00:00+00:00,2026-10-17 00:00:00+00:00


### Checks on the pre-holdout set only

From here on we only describe the **pre-holdout** rows. The holdout stays closed.

**Missing labels.** The last 7 days before the holdout are already removed, so the labels that are still missing come from coins that stopped trading while they were in the universe (delisted, or migrated to a new ticker). Those rows will drop out of every IC and every model fit, which hides what happened to the coin at the end. The numbers are small here, but they matter for the final evaluation (see `docs/10-case_study_1_audit.md`).

In [16]:
horizons = [1, 3, 7]

print("Rows in pre-holdout:", len(pre_holdout))
print()
print("Missing labels:")
print(pre_holdout[[f"fwd_log_return_{n}" for n in horizons]].isna().sum())
print()
print("Share of up days (missing labels skipped):")
print(pre_holdout[[f"fwd_direction_{n}" for n in horizons]].mean())

Rows in pre-holdout: 37725

Missing labels:
fwd_log_return_1     3
fwd_log_return_3     9
fwd_log_return_7    21
dtype: int64

Share of up days (missing labels skipped):
fwd_direction_1    0.479110
fwd_direction_3    0.472558
fwd_direction_7    0.452578
dtype: float64


The features: a coin only enters the universe on its 31st bar, so none of them should be missing.

In [17]:
features = ["past_log_return_7", "past_log_return_14", "past_log_return_30"]

print("Missing features:")
print(pre_holdout[features].isna().sum())
print()
print(pre_holdout[features].describe())

Missing features:
past_log_return_7     0
past_log_return_14    0
past_log_return_30    0
dtype: int64

       past_log_return_7  past_log_return_14  past_log_return_30
count       37725.000000        37725.000000        37725.000000
mean           -0.011287           -0.020645           -0.023513
std             0.165788            0.236982            0.384345
min            -2.359258           -2.541866           -2.874894
25%            -0.102725           -0.150895           -0.240520
50%            -0.012964           -0.027265           -0.039679
75%             0.075222            0.100216            0.159879
max             1.531704            2.021841            2.663073


### Look-ahead checks on what we are about to save

These recompute every feature and label in a second, independent way from the frozen universe and compare (`src/wh_research/evaluation/checks.py`). The same checks can be run from the shell with `uv run python scripts/check_case_study_data.py`. They read the holdout rows only to confirm that they are correctly built; nothing is described or plotted.

In [18]:
results = [("holdout split: no overlap, universe rows only, purged", check_split(pre_holdout, holdout, 7))]
for n in (7, 14, 30):
    results.append((f"past_log_return_{n} looks back {n} days", check_saved_columns(pre_holdout, df, -n, f"past_log_return_{n}")))
for n in (1, 3, 7):
    results.append((f"fwd_log_return_{n} looks ahead {n} days", check_saved_columns(pre_holdout, df, n, f"fwd_log_return_{n}")))

for name, (passed, detail) in results:
    print(f"{'PASS' if passed else 'FAIL'}  {name}  ({detail})")

assert all(passed for _, (passed, _) in results)

PASS  holdout split: no overlap, universe rows only, purged  (0 overlapping rows, 0 labels reaching into the holdout, 0 rows outside the universe)
PASS  past_log_return_7 looks back 7 days  (past_log_return_7: 37725 rows compared, 0 differ)
PASS  past_log_return_14 looks back 14 days  (past_log_return_14: 37725 rows compared, 0 differ)
PASS  past_log_return_30 looks back 30 days  (past_log_return_30: 37725 rows compared, 0 differ)
PASS  fwd_log_return_1 looks ahead 1 days  (fwd_log_return_1: 37725 rows compared, 0 differ)
PASS  fwd_log_return_3 looks ahead 3 days  (fwd_log_return_3: 37725 rows compared, 0 differ)
PASS  fwd_log_return_7 looks ahead 7 days  (fwd_log_return_7: 37725 rows compared, 0 differ)


### Saving the sets

Each set is saved next to the universe file in `data/hyperliquid/case_study_data`, with all columns (candle columns, `in_universe`, the `fwd_*` labels and the `past_log_return_*` features).

In [19]:
save_dir = PROJECT_ROOT/"data/hyperliquid/case_study_data"

for name, part in sets.items():
    part.to_parquet(save_dir/f"{name}.parquet", index=False)
    print(f"saved {name}.parquet ({len(part)} rows)")

saved pre_holdout.parquet (37725 rows)
saved holdout.parquet (21454 rows)
